In [0]:
from pyspark.sql.functions import when, col,concat_ws, substring, regexp_replace, sha2, concat, upper, lit

In [0]:
df = spark.read.csv(
    "s3://s3-lge-he-inbound-eic-dev/HEDS/1406/1406_SET Serial Number List.csv",
    header=True,
    inferSchema=True
)
display(df)

In [0]:
df = df.withColumn("LOT_MAC", upper(col("LOT_MAC")))
display(df)

In [0]:
df = df.withColumn(
    "mac_addr",
    concat_ws(":",
        substring(col("LOT_MAC"), 1, 2),
        substring(col("LOT_MAC"), 3, 2),
        substring(col("LOT_MAC"), 5, 2),
        substring(col("LOT_MAC"), 7, 2),
        substring(col("LOT_MAC"), 9, 2),
        substring(col("LOT_MAC"), 11, 2),
    )
)

display(df)

In [0]:
tv_salt = dbutils.secrets.get("admin", "salt")

df=df.withColumn("mac_addr_hashed", when(col("mac_addr").isNull() | (col("mac_addr") == ''), col("mac_addr")).otherwise(sha2(concat(col("mac_addr"), lit(tv_salt)), 256)))

display(df)

In [0]:
df.toDF(*[c.replace(" ", "_") for c in df.columns]).write.format("delta").mode("overwrite").saveAsTable("sandbox.z_yeswook_kim.heds_1406_v3")

In [0]:
%sql
select *
from sandbox.z_yeswook_kim.heds_1406_v3
where 1=1
    -- and len(LOT_MAC) != len('60756C2B85BB')

In [0]:
%sql
  select 
    mac_addr, 
    first(Cntry_CODE) as cntry_code, 
    min(crt_date) as min_crt_date
  from (
    select 
      mac_addr, 
      Cntry_CODE, 
      crt_date,
      row_number() over (partition by mac_addr order by crt_date asc) as rn
    from eic_data_private.tlamp.activation_date
  ) t
  where rn = 1
        and mac_addr='D0:CD:BF:55:68:C2'
  group by mac_addr


In [0]:
%sql
select h.*
    --    a.min_crt_date,
    --    a.cntry_code as a_cntry_code
from sandbox.z_yeswook_kim.heds_1406_v3 h
where 1=1
    and mac_addr='D0:CD:BF:55:68:C2'

In [0]:
%sql
select h.*,
       a.min_crt_date,
       a.cntry_code as a_cntry_code
from sandbox.z_yeswook_kim.heds_1406_v3 h
left join (
  select 
    mac_addr, 
    first(Cntry_CODE) as cntry_code, 
    min(crt_date) as min_crt_date
  from (
    select 
      mac_addr, 
      Cntry_CODE, 
      crt_date,
      row_number() over (partition by mac_addr order by crt_date asc) as rn
    from eic_data_private.tlamp.activation_date
    where 1=1
     and mac_addr='D0:CD:BF:55:68:C2'
  ) t
  where rn = 1
  group by mac_addr
) a
on h.mac_addr = a.mac_addr
where 1=1
    and h.mac_addr='D0:CD:BF:55:68:C2'

In [0]:
%sql
create or replace table sandbox.z_yeswook_kim.heds_1406_v4 as
select h.*,
       a.min_crt_date,
       a.cntry_code as a_cntry_code
from sandbox.z_yeswook_kim.heds_1406_v3 h
left join (
  select 
    mac_addr, 
    first(Cntry_CODE) as cntry_code, 
    min(crt_date) as min_crt_date
  from (
    select 
      mac_addr, 
      Cntry_CODE, 
      crt_date,
      row_number() over (partition by mac_addr order by crt_date asc) as rn
    from eic_data_private.tlamp.activation_date
    -- where 1=1
    --  and mac_addr='D0:CD:BF:55:68:C2'
  ) t
  where rn = 1
  group by mac_addr
) a
on h.mac_addr = a.mac_addr
order by h.no

In [0]:
%sql
select *
from sandbox.z_yeswook_kim.heds_1406_v4
where 1=1  
    -- and mac_addr='D0:CD:BF:55:68:C2'

In [0]:
%sql
select *
from eic_data_dimension.common.country_code
where 1=1
    -- and country_code = 'BD'